# IBKR All-News / BroadTape Validation

This notebook is intentionally separate from `news_api/news_module_validation.ipynb`.

Goal: validate whether this IB Gateway/account can subscribe to true BroadTape all-news feeds through NEWS contracts.

Important distinction:
- `watchlist_monitor` = contract-specific news for selected stock contracts.
- `broadtape_monitor` = all-news NEWS contracts such as `BRF:BRF_ALL@BRF`, `BZ:BZ_ALL@BZ`, `FLY:FLY_ALL@FLY`.


## 0. Setup

Run this from the repository root. IB Gateway should already be running on port `4001`.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from news_api.config import SETTINGS

print("repo_root", repo_root)
print("host", SETTINGS.host)
print("port", SETTINGS.port)
print("market_data_type", SETTINGS.market_data_type)
print("broadtape_specs", SETTINGS.broadtape_specs)


## 1. Provider Probe

`reqNewsProviders()` asks IB Gateway which API news providers are available for this username/session.

In [ ]:
RUN_PROVIDER_PROBE = True
PROVIDER_CLIENT_ID = 1100

if RUN_PROVIDER_PROBE:
    from news_api.realtime.probe import fetch_news_providers

    providers = fetch_news_providers(client_id=PROVIDER_CLIENT_ID)
    print("provider_count", len(providers))
    print("news_providers", providers)
else:
    print("Provider probe skipped.")


## 2. True BroadTape All-News Probe

This is the actual all-news test. It subscribes to `secType='NEWS'` contracts. If this returns `Valid are: []` or entitlement errors, the account/session does not currently have usable BroadTape NEWS sources.

In [ ]:
RUN_BROADTAPE_PROBE = True
BROADTAPE_CLIENT_ID = 1101
BROADTAPE_SECONDS = 30

if RUN_BROADTAPE_PROBE:
    from news_api.realtime.broadtape_monitor import run_broadtape_probe

    broadtape_client = run_broadtape_probe(
        client_id=BROADTAPE_CLIENT_ID,
        seconds=BROADTAPE_SECONDS,
        specs=SETTINGS.broadtape_specs,
        market_data_type=SETTINGS.market_data_type,
    )
    print("captured_broadtape_events", len(broadtape_client.events))
    print("captured_broadtape_errors", len(broadtape_client.errors))
    for event in broadtape_client.events[:20]:
        print("broadtape_tick_news", event.source, event.provider, event.article_id, event.headline[:120])
    for error in broadtape_client.errors[:20]:
        print("broadtape_error", error)
else:
    print("BroadTape probe skipped.")


## 3. Optional Watchlist-Specific Sanity Check

This is not all-news monitoring. It is only a sanity check for the already-working stock-contract path.

In [ ]:
RUN_WATCHLIST_SANITY_CHECK = False
WATCHLIST_CLIENT_ID = 1102
WATCHLIST_SECONDS = 30

if RUN_WATCHLIST_SANITY_CHECK:
    from news_api.realtime.watchlist_monitor import run_watchlist_probe

    watchlist_service = run_watchlist_probe(
        client_id=WATCHLIST_CLIENT_ID,
        seconds=WATCHLIST_SECONDS,
    )
    print("captured_watchlist_events", len(watchlist_service.events))
else:
    print("Watchlist sanity check skipped.")


## 4. How to Interpret Results

- `captured_broadtape_events > 0`: true all-news BroadTape is working.
- `provider_count == 0` and/or error `Valid are: []`: this IB username/session has no valid BroadTape NEWS sources.
- Watchlist events do not prove all-news works; they only prove contract-specific news works.
